In [ ]:
import matplotlib.pyplot as plt
import re
import numpy as np

In [ ]:
with open("../os/qemu-run.log") as f:
    lines = f.readlines()

In [ ]:
data = []

for l in lines:
    if "[TRACE] FrameAllocator:" in l:
        arr = re.sub(r"\x1b\[[0-9;]*m", "", l).split()
        data.append((int(arr[3]), int(arr[6]), int(arr[9]), int(arr[13])))

In [ ]:
used = np.array([x[0] for x in data])
page_cache = np.array([x[1] for x in data])
free = np.array([x[2] for x in data])
heap = np.array([x[3] for x in data])

In [ ]:
heap

In [ ]:
plt.figure(figsize=(20, 8), dpi=300)
plt.plot(range(len(data)), used / 256, label="Used (MiB)")
plt.plot(range(len(data)), page_cache / 256, label="Page Cache (MiB)")
plt.plot(range(len(data)), free / 256, label="Free (MiB)")
plt.plot(range(len(data)), heap / 256, label="Heap (MiB)")
plt.legend()
plt.yticks(range(0, 128, 5))
plt.grid(True, alpha=0.2)

In [ ]:
data[0]

In [ ]:
import numpy as np

plt.figure(figsize=(20, 4), dpi=300)
plt.plot(range(len(data)), [x[1] for x in data], label="Free")
plt.plot(range(len(data)), [x[2] for x in data], label="Recycled")
plt.plot(range(len(data)), np.array([x[1] for x in data]) + np.array([x[2] for x in data]), label="TotalFree")
plt.legend()
plt.savefig("memory_trace.png")

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# 把这里换成你 notebook 里已有的变量名
# events = lifecycle_logs
events = process_lifetime_logs

ansi_re = re.compile(r"\x1b\[[0-9;]*m")
fork_re = re.compile(r"pid\[(\d+)\] forked new process with pid (\d+) - time (\d+)")
exit_re = re.compile(r"pid\[(\d+)\] sys_exit - time (\d+)")

proc_instances = {}
alive = {}
rows = []
edges = []

def new_instance(pid, parent_pid, start_time, line_no):
    gen = proc_instances.get(pid, 0) + 1
    proc_instances[pid] = gen
    key = (pid, gen)

    parent_key = alive.get(parent_pid)
    alive[pid] = key

    rows.append({
        "key": key,
        "pid": pid,
        "gen": gen,
        "label": f"pid {pid}#{gen}" if gen > 1 else f"pid {pid}",
        "parent_pid": parent_pid,
        "parent_key": parent_key,
        "start": start_time,
        "end": None,
        "fork_line": line_no,
        "exit_line": None,
    })

    if parent_key is not None:
        edges.append((parent_key, key, start_time))

    return key

# pid 0 是根进程：如果日志中没有它的 fork 事件，就从最早事件时间开始画
all_times = []
for _, msg in events:
    msg = ansi_re.sub("", msg)
    m1 = fork_re.search(msg)
    m2 = exit_re.search(msg)
    if m1:
        all_times.append(int(m1.group(3)))
    elif m2:
        all_times.append(int(m2.group(2)))

t0 = min(all_times)
new_instance(pid=0, parent_pid=None, start_time=t0, line_no=None)

for line_no, raw_msg in events:
    msg = ansi_re.sub("", raw_msg)

    m = fork_re.search(msg)
    if m:
        parent_pid = int(m.group(1))
        child_pid = int(m.group(2))
        t = int(m.group(3))
        new_instance(child_pid, parent_pid, t, line_no)
        continue

    m = exit_re.search(msg)
    if m:
        pid = int(m.group(1))
        t = int(m.group(2))

        key = alive.pop(pid, None)
        if key is None:
            # 遇到 exit 但没有对应 fork，保留为未知起点实例
            key = new_instance(pid=pid, parent_pid=None, start_time=t0, line_no=None)
            alive.pop(pid, None)

        for row in rows:
            if row["key"] == key:
                row["end"] = t
                row["exit_line"] = line_no
                break

df = pd.DataFrame(rows)

# 没有 exit 的进程画到最后一个事件时间
t_end = max(all_times)
df["end"] = df["end"].fillna(t_end)
df["duration"] = df["end"] - df["start"]

# 用相对时间，避免横轴数字过大
df["start_rel"] = (df["start"] - t0) / 1e9
df["end_rel"] = (df["end"] - t0) / 1e9
df["duration_rel"] = df["duration"] / 1e9

# 按创建时间排序，也可以改成按 pid/gen 排序
df = df.sort_values(["start", "pid", "gen"]).reset_index(drop=True)
ypos = {key: i for i, key in enumerate(df["key"])}

plt.style.use("seaborn-v0_8")

fig_h = max(8, 0.32 * len(df))
fig, ax = plt.subplots(figsize=(16, fig_h), dpi=300)

colors = cm.get_cmap("viridis", max(1, df["pid"].nunique()))
pid_to_color_idx = {pid: i for i, pid in enumerate(sorted(df["pid"].unique()))}

for _, row in df.iterrows():
    y = ypos[row["key"]]
    color = colors(pid_to_color_idx[row["pid"]] % 20)

    ax.barh(
        y=y,
        width=row["duration_rel"],
        left=row["start_rel"],
        height=0.62,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        alpha=0.85,
    )

    ax.text(
        row["start_rel"],
        y,
        f" {row['label']}",
        va="center",
        ha="left",
        fontsize=8,
        color="black",
    )

# 画父子关系：从父进程生命周期条连接到子进程 fork 起点
for parent_key, child_key, fork_time in edges:
    if parent_key not in ypos or child_key not in ypos:
        continue

    x = (fork_time - t0) / 1e9
    y_parent = ypos[parent_key]
    y_child = ypos[child_key]

    ax.plot(
        [x, x],
        [y_parent, y_child],
        color="gray",
        linewidth=0.7,
        alpha=0.55,
        zorder=0,
    )
    ax.scatter(
        [x],
        [y_child],
        marker=">",
        s=18,
        color="black",
        alpha=0.65,
        zorder=3,
    )

ax.set_yticks(range(len(df)))
ax.set_yticklabels(df["label"])
ax.invert_yaxis()

ax.set_xlabel(f"time since first lifecycle event / 1e9 ticks, base={t0}")
ax.set_ylabel("process instance")
ax.set_title("Process lifecycle Gantt chart with parent-child relations")

ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.margins(x=0.01)
plt.tight_layout()
plt.show()

In [ ]:
# 基于上一段代码得到的 df 统计最大同时存活进程数
points = []

for _, row in df.iterrows():
    points.append((row["start"], +1, row["label"]))
    points.append((row["end"], -1, row["label"]))

# 同一时间点先处理 exit，再处理 fork，避免 end==start 时虚增并发数
points.sort(key=lambda x: (x[0], x[1]))

running = 0
max_running = 0
max_times = []

for t, delta, label in points:
    running += delta
    if running > max_running:
        max_running = running
        max_times = [t]
    elif running == max_running:
        max_times.append(t)

print("最多同时运行进程数:", max_running)
print("首次达到该数量的时间:", max_times[0])
print("相对起始时间:", (max_times[0] - t0) / 1e9)